# Feature Selection and cleaning

In [51]:
# -------------------------------
# 1. Imports
# -------------------------------
import pandas as pd
import numpy as np


In [52]:
# -------------------------------
# 2. Load Dataset
# -------------------------------
url = "../Data/owid-energy-data-updated.csv"
df = pd.read_csv(url)

print(df.shape)
print(list(df.columns))
df.isna().mean().sort_values(ascending=False)

(14384, 130)
['country', 'year', 'iso_code', 'population', 'biofuel_cons_change_pct', 'biofuel_cons_change_twh', 'biofuel_cons_per_capita', 'biofuel_consumption', 'biofuel_elec_per_capita', 'biofuel_electricity', 'biofuel_share_elec', 'biofuel_share_energy', 'carbon_intensity_elec', 'coal_cons_change_pct', 'coal_cons_change_twh', 'coal_cons_per_capita', 'coal_consumption', 'coal_elec_per_capita', 'coal_electricity', 'coal_prod_change_pct', 'coal_prod_change_twh', 'coal_prod_per_capita', 'coal_production', 'coal_share_elec', 'coal_share_energy', 'electricity_demand', 'electricity_demand_per_capita', 'electricity_generation', 'electricity_share_energy', 'energy_cons_change_pct', 'energy_cons_change_twh', 'energy_per_capita', 'energy_per_gdp', 'fossil_cons_change_pct', 'fossil_cons_change_twh', 'fossil_elec_per_capita', 'fossil_electricity', 'fossil_energy_per_capita', 'fossil_fuel_consumption', 'fossil_share_elec', 'fossil_share_energy', 'gas_cons_change_pct', 'gas_cons_change_twh', 'gas

biofuel_cons_change_pct             0.919494
nuclear_cons_change_pct             0.902461
solar_cons_change_pct               0.896065
wind_cons_change_pct                0.885498
other_renewables_cons_change_pct    0.836902
                                      ...   
oil_production                      0.187848
iso_code                            0.019118
population                          0.006257
year                                0.000000
country                             0.000000
Length: 130, dtype: float64

In [53]:
selected_columns = [
    "country", "year", "iso_code", "population",
    "energy_per_capita",
    "gdp",
    "fossil_share_energy", "coal_share_energy", "gas_share_energy",
    "oil_share_energy", "renewables_share_energy", "low_carbon_share_energy",
    "nuclear_share_energy", "biofuel_share_energy", "hydro_share_energy",
    "solar_share_energy", "wind_share_energy",
    "greenhouse_gas_emissions"
]

df_selected = df[selected_columns].copy()

df_selected['gdp_per_capita'] = df_selected['gdp'] / df_selected['population']
df_selected.drop(columns=['gdp'], inplace=True)

df_selected['ghg_per_capita'] = df_selected['greenhouse_gas_emissions'] / df_selected['population']
df_selected.drop(columns=['greenhouse_gas_emissions'], inplace=True)

# these columns aren't linearly distributed, so we apply log transformation to make them linear
skewed_cols = ["gdp_per_capita", "population", "ghg_per_capita"]
for col in skewed_cols:
    df_selected[f"log_{col}"] = np.log1p(df_selected[col])
    df_selected.drop(columns=[col], inplace=True)
    


df_selected.head()

,country,year,iso_code,energy_per_capita,fossil_share_energy,coal_share_energy,gas_share_energy,oil_share_energy,renewables_share_energy,low_carbon_share_energy,nuclear_share_energy,biofuel_share_energy,hydro_share_energy,solar_share_energy,wind_share_energy,log_gdp_per_capita,log_population,log_ghg_per_capita
0,Afghanistan,1900,AFG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.364720,NaN
1,Afghanistan,1901,AFG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.373903,NaN
2,Afghanistan,1902,AFG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.384647,NaN
3,Afghanistan,1903,AFG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.396926,NaN
4,Afghanistan,1904,AFG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.409204,NaN


In [54]:
df_selected.groupby("year").apply(lambda x: x.isna().mean().mean())

/tmp/ipykernel_20795/1718478326.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_selected.groupby("year").apply(lambda x: x.isna().mean().mean())


year
1900    0.779133
1901    0.780488
1902    0.780488
1903    0.780488
1904    0.780488
          ...   
2020    0.368801
2021    0.369115
2022    0.369115
2023    0.377276
2024    0.246399
Length: 125, dtype: float64

In [55]:
df_selected.groupby("country").apply(lambda x: x.isna().mean().mean()).sort_values(ascending=False)

/tmp/ipykernel_20795/3366396661.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_selected.groupby("country").apply(lambda x: x.isna().mean().mean()).sort_values(ascending=False)


country
Curacao            0.754630
Greenland          0.743728
Afghanistan        0.736559
Mozambique         0.732444
Albania            0.727599
                     ...   
Uzbekistan         0.025000
Turkmenistan       0.025000
Croatia            0.015873
North Macedonia    0.015873
Slovenia           0.015873
Length: 177, dtype: float64

In [56]:
unique_countries = df_selected["country"].unique()
print(unique_countries)

['Afghanistan' 'Albania' 'Algeria' 'American Samoa' 'Angola'
 'Antigua and Barbuda' 'Argentina' 'Armenia' 'Aruba' 'Australia' 'Austria'
 'Azerbaijan' 'Bahrain' 'Bangladesh' 'Barbados' 'Belarus' 'Belgium'
 'Belize' 'Benin' 'Bermuda' 'Bhutan' 'Bolivia' 'Bosnia and Herzegovina'
 'Botswana' 'Brazil' 'Bulgaria' 'Burkina Faso' 'Burundi' 'Cambodia'
 'Cameroon' 'Canada' 'Cayman Islands' 'Central African Republic' 'Chad'
 'Chile' 'China' 'Colombia' 'Comoros' 'Costa Rica' "Cote d'Ivoire"
 'Croatia' 'Cuba' 'Curacao' 'Cyprus' 'Czechia' 'Denmark' 'Djibouti'
 'Dominica' 'Dominican Republic' 'Ecuador' 'El Salvador'
 'Equatorial Guinea' 'Eritrea' 'Estonia' 'Eswatini' 'Ethiopia'
 'Faroe Islands' 'Fiji' 'Finland' 'France' 'French Polynesia' 'Gabon'
 'Georgia' 'Germany' 'Ghana' 'Greece' 'Greenland' 'Grenada' 'Guam'
 'Guatemala' 'Guinea' 'Guinea-Bissau' 'Guyana' 'Haiti' 'Honduras'
 'Hungary' 'Iceland' 'India' 'Indonesia' 'Iraq' 'Ireland' 'Israel' 'Italy'
 'Jamaica' 'Japan' 'Jordan' 'Kazakhstan' 'Kenya' 'K

In [57]:
# -------------------------------
# 4. Filter by Year and Country
# -------------------------------
df_selected = df_selected[df_selected["year"] >= 1965]

# Drop aggregates. Non-aggregate entries have 3-letter ISO codes
df_selected = df_selected[df_selected['iso_code'].str.len() == 3]

# Drop countries with >40% missing
threshold = 0.4
country_missing = df_selected.groupby("country").apply(lambda x: x.isna().mean().mean())
good_countries = country_missing[country_missing <= threshold].index
df_selected = df_selected[df_selected["country"].isin(good_countries)]

/tmp/ipykernel_20795/1382756241.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  country_missing = df_selected.groupby("country").apply(lambda x: x.isna().mean().mean())


In [58]:
# -------------------------------
# 5. Drop rows with missing target
# -------------------------------
df_selected = df_selected.dropna(subset=['energy_per_capita']).copy()

In [59]:
df_selected.isna().mean().sort_values(ascending=False)

log_ghg_per_capita         0.559588
log_gdp_per_capita         0.041441
solar_share_energy         0.001544
hydro_share_energy         0.001544
coal_share_energy          0.001544
fossil_share_energy        0.001544
renewables_share_energy    0.001544
low_carbon_share_energy    0.001544
oil_share_energy           0.001544
gas_share_energy           0.001544
wind_share_energy          0.001544
biofuel_share_energy       0.001544
nuclear_share_energy       0.001544
country                    0.000000
year                       0.000000
iso_code                   0.000000
energy_per_capita          0.000000
log_population             0.000000
dtype: float64

In [60]:
# -------------------------------
# 6. Missing Value Imputation
# -------------------------------
df_selected = df_selected.sort_values(['country', 'year'])

# Interpolate GDP per capita
df_selected[["log_gdp_per_capita"]] = df_selected.groupby("country")[["log_gdp_per_capita"]].transform(
    lambda x: x.interpolate().ffill().bfill()
)

# Forward/backward fill all share columns + emissions
share_cols = [
    "fossil_share_energy", "coal_share_energy", "gas_share_energy",
    "oil_share_energy", "renewables_share_energy", "low_carbon_share_energy",
    "nuclear_share_energy", "biofuel_share_energy", "hydro_share_energy",
    "solar_share_energy", "wind_share_energy", "log_ghg_per_capita"
]
df_selected[share_cols] = df_selected.groupby("country")[share_cols].transform(
    lambda x: x.ffill().bfill()
)

In [61]:
# -------------------------------
# 7. Final Check
# -------------------------------
print("Missing values per column after imputation:")
print(df_selected.isna().sum())

print("\nDataset shape:", df_selected.shape)
print("Number of countries:", df_selected['country'].nunique())

Missing values per column after imputation:
country                    0
year                       0
iso_code                   0
energy_per_capita          0
fossil_share_energy        0
coal_share_energy          0
gas_share_energy           0
oil_share_energy           0
renewables_share_energy    0
low_carbon_share_energy    0
nuclear_share_energy       0
biofuel_share_energy       0
hydro_share_energy         0
solar_share_energy         0
wind_share_energy          0
log_gdp_per_capita         0
log_population             0
log_ghg_per_capita         0
dtype: int64

Dataset shape: (3885, 18)
Number of countries: 69


In [62]:
df_selected.to_csv("../Data/owid-energy-data-clean-new.csv", index=False)